# OrbitDesk router: QLoRA fine-tune of Qwen2.5-0.5B-Instruct (free Colab T4)

Goal: replace the prompt-based router (Groq gpt-oss-20b) with a small local model that outputs
**strict JSON** `{"intent": ..., "urgency": ...}`, then compare on the same 60-example held-out test set.

**How to run**
1. Runtime > Change runtime type > **T4 GPU**.
2. Optional: add a Colab secret `HF_TOKEN` (write access) and set `HF_PUSH_REPO` below to push the adapter.
3. Provide the data: set `RAW_BASE` to your GitHub repo's raw URL, or leave it empty and upload
   `finetune/data/train.jsonl` and `finetune/data/test.jsonl` when prompted.
4. Run all. Download `finetuned_predictions.json` at the end and put it in `finetune/results/`
   in the repo, then run `python finetune/compare_routers.py`.

The code cells below are generated from `finetune/router_train.py` and `finetune/router_metrics.py`
by `finetune/build_notebook.py`, so this notebook runs the same code that was smoke-tested locally.

In [ ]:
# Pinned to the versions the local smoke test used (trl 1.15 changed the SFT loss path).
!pip install -q "trl==1.14.2" "peft>=0.12" "accelerate>=1.0" "datasets>=2.20" bitsandbytes pyyaml

In [ ]:
import os
from pathlib import Path

RAW_BASE = ""          # e.g. "https://raw.githubusercontent.com/<you>/orbitdesk/main"
HF_PUSH_REPO = ""      # e.g. "<your-hf-username>/orbitdesk-router-lora" (needs HF_TOKEN secret)

Path("data").mkdir(exist_ok=True)
if RAW_BASE:
    for name in ("train.jsonl", "test.jsonl"):
        os.system(f"wget -q -O data/{name} {RAW_BASE}/finetune/data/{name}")
else:
    from google.colab import files
    print("Upload train.jsonl and test.jsonl")
    for name, content in files.upload().items():
        Path("data", name).write_bytes(content)
print({p.name: sum(1 for _ in open(p)) for p in Path("data").glob("*.jsonl")})

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret: the adapter will not be pushed to the Hub.")

In [ ]:
%%writefile router_metrics.py
"""Router metrics, dependency-free (shared by the Colab notebook and compare_routers.py)."""

import json
import re

INTENTS = ["policy_question", "action", "small_talk", "out_of_scope"]
URGENCIES = ["low", "medium", "high"]


def parse_strict(text: str) -> dict | None:
    """Strict: the WHOLE output must be one JSON object with valid labels (that is the contract)."""
    try:
        d = json.loads(text.strip())
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(d, dict) or d.get("intent") not in INTENTS or d.get("urgency") not in URGENCIES:
        return None
    return {"intent": d["intent"], "urgency": d["urgency"]}


def parse_lenient(text: str) -> dict | None:
    """Lenient: first {...} anywhere in the output (what the production router accepts)."""
    m = re.search(r"\{.*?\}", text or "", re.S)
    return parse_strict(m.group(0)) if m else None


def macro_f1(gold: list[str], pred: list[str], labels: list[str]) -> float:
    f1s = []
    for lab in labels:
        tp = sum(g == lab and p == lab for g, p in zip(gold, pred, strict=True))
        fp = sum(g != lab and p == lab for g, p in zip(gold, pred, strict=True))
        fn = sum(g == lab and p != lab for g, p in zip(gold, pred, strict=True))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return sum(f1s) / len(f1s)


def score(gold_rows: list[dict], raw_outputs: list[str]) -> dict:
    """Unparseable outputs count as wrong (predicted label "INVALID")."""
    strict = [parse_strict(o) for o in raw_outputs]
    lenient = [parse_lenient(o) for o in raw_outputs]
    pi = [(p or {}).get("intent", "INVALID") for p in lenient]
    pu = [(p or {}).get("urgency", "INVALID") for p in lenient]
    gi = [g["intent"] for g in gold_rows]
    gu = [g["urgency"] for g in gold_rows]
    n = len(gold_rows)
    return {
        "n": n,
        "strict_json_rate": round(sum(p is not None for p in strict) / n, 4),
        "intent_accuracy": round(sum(a == b for a, b in zip(gi, pi, strict=True)) / n, 4),
        "intent_macro_f1": round(macro_f1(gi, pi, INTENTS), 4),
        "urgency_accuracy": round(sum(a == b for a, b in zip(gu, pu, strict=True)) / n, 4),
        "urgency_macro_f1": round(macro_f1(gu, pu, URGENCIES), 4),
        "joint_accuracy": round(sum(a == b and c == d for a, b, c, d in zip(gi, pi, gu, pu, strict=True)) / n, 4),
    }

In [ ]:
%%writefile router_train.py
"""QLoRA fine-tuning of a small instruct model as the OrbitDesk router (strict JSON {intent, urgency}).

This file IS the notebook: finetune/build_notebook.py embeds it into router_lora.ipynb,
so the code that runs on Colab is the code that was smoke-tested locally.

    Colab (T4):  run the notebook (QLoRA 4-bit, full data, ~3 epochs)
    Local smoke: uv run --group finetune python finetune/router_train.py --smoke
                 (CPU, no 4-bit, tiny data, 2 steps: checks the code path, not quality)
"""

import argparse
import json
import os
import statistics
import time
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ROUTER_SYSTEM = 'You classify messages sent to OrbitDesk, the internal HR and IT helpdesk of Orbit Systems.\nintent:\n- policy_question: asks what a company HR/IT/finance policy or IT runbook says, or how to fix/do something (leave rules, VPN errors, expenses, onboarding, passwords).\n- action: wants something done or looked up in a system: check MY leave balance, raise/create/open a ticket, check a ticket\'s status.\n- small_talk: greetings, thanks, chit-chat, questions about the assistant itself.\n- out_of_scope: anything unrelated to Orbit HR/IT work (general knowledge, coding help, jokes, personal advice), or requests to do something harmful or forbidden.\nurgency:\n- high: work is fully blocked, security incident (lost/stolen device, phishing, breach), outage affecting many people.\n- medium: degraded but a workaround exists, or a deadline soon.\n- low: everything else, including all small_talk and out_of_scope.\nReply with ONLY a JSON object: {"intent": "<intent>", "urgency": "<urgency>"}\n'  # inlined from prompts/router_v1.yaml
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def router_system() -> str:
    if ROUTER_SYSTEM:
        return ROUTER_SYSTEM
    import yaml

    return yaml.safe_load((HERE.parent / "prompts" / "router_v1.yaml").read_text(encoding="utf-8"))["system"]


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]


def to_prompt(message: str) -> list[dict]:
    """Same system text + user template as the production prompt router (zero-shot form)."""
    return [{"role": "system", "content": router_system()}, {"role": "user", "content": f"Message: {message}"}]


def to_example(row: dict) -> dict:
    target = json.dumps({"intent": row["intent"], "urgency": row["urgency"]})
    return {"prompt": to_prompt(row["message"]), "completion": [{"role": "assistant", "content": target}]}


def load_model(model_name: str, use_4bit: bool):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name)
    kwargs = {}
    if use_4bit:  # QLoRA: frozen base in 4-bit NF4, LoRA adapters trained in bf16/fp16
        from transformers import BitsAndBytesConfig

        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16
        )
        kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    return model, tok


def train(
    train_rows,
    eval_rows,
    out_dir: str,
    use_4bit: bool,
    epochs: float = 3,
    max_steps: int = -1,
    model_name: str = BASE_MODEL,
):
    import math

    from datasets import Dataset
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTConfig, SFTTrainer

    model, tok = load_model(model_name, use_4bit)
    steps_per_epoch = math.ceil(len(train_rows) / 16)  # batch 8 x grad accumulation 2
    warmup = 0 if max_steps > 0 else max(1, int(0.05 * steps_per_epoch * epochs))  # ~5% warmup
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    args = SFTConfig(
        output_dir=out_dir,
        num_train_epochs=epochs,
        max_steps=max_steps,
        # smoke runs on CPU: tiny batches (a 151k-token vocabulary makes fp32 logits huge)
        per_device_train_batch_size=2 if max_steps > 0 else 8,
        gradient_accumulation_steps=1 if max_steps > 0 else 2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=warmup,  # transformers 5 removed warmup_ratio
        logging_steps=1 if max_steps > 0 else 10,
        eval_strategy="epoch" if max_steps < 0 else "no",
        save_strategy="no",
        max_length=512,
        completion_only_loss=True,  # loss only on the JSON answer, not the system prompt
        fp16=use_4bit,  # T4: fp16 mixed precision
        bf16=False,  # TRL defaults to bf16, which the T4 (and CPU smoke runs) can't use
        use_cpu=not use_4bit,
        report_to="none",
        seed=13,
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=Dataset.from_list([to_example(r) for r in train_rows]),
        eval_dataset=Dataset.from_list([to_example(r) for r in eval_rows]) if max_steps < 0 else None,
        processing_class=tok,
        peft_config=lora,
    )
    t0 = time.perf_counter()
    trainer.train()
    train_s = time.perf_counter() - t0
    trainer.model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    return trainer.model, tok, train_s


def predict(model, tok, messages: list[str]) -> tuple[list[str], list[float]]:
    """Greedy decoding, one message at a time (that's how the router runs in production)."""
    import torch

    model.eval()
    outs, lat = [], []
    for m in messages:
        # transformers 5 returns a BatchEncoding (input_ids + attention_mask) here, not a bare tensor
        enc = tok.apply_chat_template(
            to_prompt(m), add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        ids = enc["input_ids"]
        t0 = time.perf_counter()
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=24, do_sample=False, pad_token_id=tok.eos_token_id)
        lat.append((time.perf_counter() - t0) * 1000)
        outs.append(tok.decode(gen[0][ids.shape[1] :], skip_special_tokens=True))
    return outs, lat


def run(data_dir: Path, out_dir: str, use_4bit: bool, smoke: bool = False, push_to: str = "") -> dict:
    from router_metrics import score

    train_rows, test_rows = load_jsonl(data_dir / "train.jsonl"), load_jsonl(data_dir / "test.jsonl")
    if smoke:
        train_rows, test_rows = train_rows[:16], test_rows[:4]
    model, tok, train_s = train(train_rows, test_rows, out_dir, use_4bit, max_steps=2 if smoke else -1)
    outputs, lat = predict(model, tok, [r["message"] for r in test_rows])
    metrics = score(test_rows, outputs)
    import torch

    result = {
        "base_model": BASE_MODEL,
        "method": "QLoRA 4-bit NF4, LoRA r=16 alpha=32 on all linear layers" if use_4bit else "LoRA (no 4-bit)",
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "smoke_test": smoke,
        "n_train": len(train_rows),
        "train_seconds": round(train_s, 1),
        "metrics": metrics,
        "latency_ms_p50": round(statistics.median(lat), 1),
        "latency_ms_p95": round(sorted(lat)[int(0.95 * (len(lat) - 1))], 1),
        "predictions": [
            {
                "message": r["message"],
                "gold": {"intent": r["intent"], "urgency": r["urgency"]},
                "output": o,
                "latency_ms": round(ms, 1),
            }
            for r, o, ms in zip(test_rows, outputs, lat, strict=True)
        ],
    }
    if push_to:
        model.push_to_hub(push_to, private=False)
        tok.push_to_hub(push_to)
        result["hub_repo"] = push_to
    return result


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--data", default=str(HERE / "data"))
    ap.add_argument("--out", default=str(HERE / "outputs" / "router-lora"))
    args = ap.parse_args()
    data = Path(args.data)
    if args.smoke and not (data / "train.jsonl").exists():
        data = HERE / "data_smoke"  # tiny hand-written set so the code path runs before any LLM data exists
    import torch

    res = run(
        data,
        args.out,
        use_4bit=torch.cuda.is_available(),
        smoke=args.smoke,
        push_to=os.environ.get("HF_PUSH_REPO", "") if os.environ.get("HF_TOKEN") else "",
    )
    print(json.dumps({k: v for k, v in res.items() if k != "predictions"}, indent=2))
    name = "finetuned_smoke.json" if args.smoke else "finetuned_predictions.json"
    (HERE / "results").mkdir(exist_ok=True)
    (HERE / "results" / name).write_text(json.dumps(res, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
import json, torch
import router_train

assert torch.cuda.is_available(), "Select a T4 GPU runtime"
push = HF_PUSH_REPO if os.environ.get("HF_TOKEN") and HF_PUSH_REPO else ""
if push:
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
res = router_train.run(Path("data"), "outputs/router-lora", use_4bit=True, smoke=False, push_to=push)
Path("finetuned_predictions.json").write_text(json.dumps(res, indent=2))
print(json.dumps({k: v for k, v in res.items() if k != "predictions"}, indent=2))

### Read the result honestly
Compare `metrics` above with the Groq rows from `compare_routers.py`. A fine-tuned 0.5B model often
matches a large model on a narrow 4-way classification and wins on latency and cost per request,
but it can lose on messages unlike the synthetic training data. Report whatever happened.

In [ ]:
from google.colab import files
files.download("finetuned_predictions.json")